In [ ]:
# Improved Digit Recognition + Interactive Correction
# ---------------------------------------------------
# Requirements:
#   pip install tensorflow opencv-python matplotlib numpy
#
# Usage:
#   - Put your image at /content/output.jpeg
#   - Run this script in terminal, Jupyter, or Colab.

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, models, layers
import sys

# ---------------------------
# Config
# ---------------------------
IMAGE_CANDIDATES = [
    "/content/output.jpeg",  # ✅ updated path
    "/mnt/data/internship  output.jpeg",
    r"C:\Users\chith\OneDrive\Documents\Desktop\internship  output.jpeg"
]
MODEL_FILE = "digit_recognition_model.h5"
TRAIN_IF_NO_MODEL = True
MNIST_EPOCHS = 3
FINETUNE_AFTER_CORRECTION = False
FINETUNE_EPOCHS = 3
MIN_BOX_WH = 12
SPLIT_W_OVER_H = 1.4
VERBOSE = True

# ---------------------------
# Helpers: model load/train
# ---------------------------
def load_or_train_model(model_file=MODEL_FILE):
    if os.path.exists(model_file):
        if VERBOSE: print("Loading model:", model_file)
        return tf.keras.models.load_model(model_file)
    if not TRAIN_IF_NO_MODEL:
        raise FileNotFoundError("Model not found and TRAIN_IF_NO_MODEL is False.")
    if VERBOSE: print("No model found. Training base CNN on MNIST (epochs =", MNIST_EPOCHS, ")")
    (x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()
    x_train = (x_train.astype("float32") / 255.0).reshape(-1,28,28,1)
    x_test  = (x_test.astype("float32") / 255.0).reshape(-1,28,28,1)

    model = models.Sequential([
        layers.Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(128,activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(10,activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train, epochs=MNIST_EPOCHS, validation_data=(x_test,y_test), verbose=2)
    model.save(model_file)
    if VERBOSE: print("Base model trained and saved.")
    return model

# ---------------------------
# Preprocessing + segmentation
# ---------------------------
def preprocess_gray(gray):
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=1)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=1)
    return th

def find_contours_sorted(binary):
    cnts_info = cv2.findContours(binary.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(cnts_info) == 3:
        _, contours, _ = cnts_info
    else:
        contours, _ = cnts_info
    boxes = []
    for c in contours:
        x,y,w,h = cv2.boundingRect(c)
        if w < MIN_BOX_WH or h < MIN_BOX_WH:
            continue
        boxes.append((x,y,w,h))
    boxes = sorted(boxes, key=lambda b: b[0])
    return boxes

def make_28x28_centered(roi, size=28, margin=4):
    if roi is None or roi.size==0:
        return np.zeros((size,size), dtype=np.uint8)
    h,w = roi.shape
    target = size - 2*margin
    scale = target / max(h,w)
    neww = max(1, int(w*scale))
    newh = max(1, int(h*scale))
    resized = cv2.resize(roi, (neww,newh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((size,size), dtype=np.uint8)
    sx = (size - neww)//2
    sy = (size - newh)//2
    canvas[sy:sy+newh, sx:sx+neww] = resized
    M = cv2.moments(canvas)
    if M["m00"] != 0:
        cx = int(M["m10"]/M["m00"])
        cy = int(M["m01"]/M["m00"])
        shiftx = size//2 - cx
        shifty = size//2 - cy
        T = np.float32([[1,0,shiftx],[0,1,shifty]])
        canvas = cv2.warpAffine(canvas, T, (size,size), borderValue=0)
    canvas = cv2.GaussianBlur(canvas, (3,3), 0)
    return canvas

def split_by_vertical_projection(roi_bin):
    h,w = roi_bin.shape
    col_sum = np.sum(roi_bin > 0, axis=0)
    window = max(3, w//30)
    kernel = np.ones(window)/window
    smooth = np.convolve(col_sum, kernel, mode='same')
    thresh = max(2, 0.18 * np.max(smooth))
    valleys = np.where(smooth < thresh)[0]
    if valleys.size == 0:
        return [roi_bin]
    splits = []
    i = 0
    while i < len(valleys):
        j = i
        while j+1 < len(valleys) and valleys[j+1] == valleys[j] + 1:
            j += 1
        mid = int((valleys[i] + valleys[j]) / 2)
        splits.append(mid)
        i = j + 1
    xs = [0] + splits + [w]
    parts = []
    for a,b in zip(xs[:-1], xs[1:]):
        if b - a <= 4: continue
        part = roi_bin[:, a:b]
        if np.sum(part) < 20: continue
        parts.append(part)
    if len(parts) == 0:
        return [roi_bin]
    return parts

# ---------------------------
# High-level pipeline
# ---------------------------
def process_image_and_predict(image_path):
    img_path = None
    for p in IMAGE_CANDIDATES:
        if os.path.exists(p):
            img_path = p
            break
    if img_path is None:
        img_path = image_path
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"No image found. Tried candidates:\n" + "\n".join(IMAGE_CANDIDATES))
    if VERBOSE: print("Using image:", img_path)
    bgr = cv2.imread(img_path)
    if bgr is None:
        raise FileNotFoundError("cv2 failed to read image.")
    gray_orig = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    binary = preprocess_gray(gray_orig)
    boxes = find_contours_sorted(binary)
    if VERBOSE: print("Initial boxes found:", len(boxes))
    expanded = []
    H,W = binary.shape
    for (x,y,w,h) in boxes:
        pad = max(2, int(0.08 * max(w,h)))
        xa = max(0, x-pad); ya = max(0, y-pad)
        xb = min(W, x+w+pad); yb = min(H, y+h+pad)
        expanded.append((xa, ya, xb-xa, yb-ya))
    boxes = expanded
    final_boxes = []
    for (x,y,w,h) in boxes:
        if w / float(h) > SPLIT_W_OVER_H and w > 28:
            roi = binary[y:y+h, x:x+w]
            parts = split_by_vertical_projection(roi)
            if len(parts) > 1:
                start = x
                for p in parts:
                    pw = p.shape[1]
                    final_boxes.append((start, y, pw, h))
                    start += pw
                continue
        final_boxes.append((x,y,w,h))
    final_boxes = sorted(final_boxes, key=lambda b: b[0])
    if VERBOSE: print("Final boxes after splitting:", len(final_boxes))
    rois_28, rois_boxes = [], []
    for (x,y,w,h) in final_boxes:
        roi = binary[y:y+h, x:x+w]
        roi28 = make_28x28_centered(roi, size=28, margin=4)
        rois_28.append(roi28)
        rois_boxes.append((x,y,w,h))
    if len(rois_28) == 0:
        print("No digit ROIs extracted.")
        return "", bgr
    model = load_or_train_model(MODEL_FILE)
    X = np.array(rois_28).astype("float32")/255.0
    X = X.reshape(-1,28,28,1)
    preds = model.predict(X, verbose=0)
    pred_labels = [int(np.argmax(p)) for p in preds]
    if VERBOSE: print("Model predicted:", pred_labels)
    annotated = bgr.copy()
    for (x,y,w,h), lbl in zip(rois_boxes, pred_labels):
        cv2.rectangle(annotated, (x,y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(annotated, str(lbl), (x, y-6), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)
    pred_string = "".join(str(l) for l in pred_labels)
    return pred_string, annotated, rois_28, rois_boxes, model

# ---------------------------
# Interactive correction
# ---------------------------
def interactive_correction(rois_28, initial_preds):
    corrected = list(str(x) for x in initial_preds)
    print("\nInteractive correction: For each crop, press Enter to accept prediction,")
    print("or type correct digit (0-9) and press Enter.")
    for i, roi in enumerate(rois_28):
        plt.figure(figsize=(2,2))
        plt.imshow(roi, cmap='gray')
        plt.title(f"Index {i} — Predicted: {initial_preds[i]}")
        plt.axis('off')
        plt.show()
        s = input(f"Accept predicted [{initial_preds[i]}]? (Enter to accept or type correct digit): ").strip()
        if s == "":
            continue
        if len(s) == 1 and s.isdigit():
            corrected[i] = s
        else:
            print("Invalid input; keeping original.")
    corrected_string = "".join(corrected)
    return corrected, corrected_string

# ---------------------------
# Fine-tune on corrected labels (optional)
# ---------------------------
def finetune_on_corrections(model, rois_28, corrected_labels, epochs=FINETUNE_EPOCHS):
    X_aug, y_aug = [], []
    for roi, lbl in zip(rois_28, corrected_labels):
        lab = int(lbl)
        base = roi.astype('uint8')
        X_aug.append(base); y_aug.append(lab)
        for angle in (-10, -5, 5, 10):
            M = cv2.getRotationMatrix2D((14,14), angle, 1.0)
            aug = cv2.warpAffine(base, M, (28,28), borderValue=0)
            X_aug.append(aug); y_aug.append(lab)
    X_aug = np.array(X_aug).astype('float32')/255.0
    X_aug = X_aug.reshape(-1,28,28,1)
    y_aug = np.array(y_aug)
    (x_train, y_train), _ = datasets.mnist.load_data()
    x_train = x_train.astype('float32')/255.0
    x_train = x_train.reshape(-1,28,28,1)
    X_comb = np.concatenate([x_train, X_aug], axis=0)
    y_comb = np.concatenate([y_train, y_aug], axis=0)
    idx = np.arange(len(X_comb)); np.random.shuffle(idx)
    X_comb = X_comb[idx]; y_comb = y_comb[idx]
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_comb, y_comb, epochs=epochs, batch_size=128, verbose=2)
    model.save(MODEL_FILE)
    return model

# ---------------------------
# Main entry
# ---------------------------
if __name__ == "__main__":
    img_path = None
    for p in IMAGE_CANDIDATES:
        if os.path.exists(p):
            img_path = p
            break
    if img_path is None:
        if len(sys.argv) > 1 and os.path.exists(sys.argv[1]):
            img_path = sys.argv[1]
        else:
            print("No image found in candidates. Provide path as first argument.")
            sys.exit(1)
    pred_string, annotated, rois_28, rois_boxes, model = process_image_and_predict(img_path)
    print("\nInitial predicted string:", pred_string)
    plt.figure(figsize=(8,6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title("Initial predictions (green)")
    plt.axis('off')
    plt.show()
    corrected_list, corrected_string = interactive_correction(rois_28, list(pred_string))
    print("\nCorrected string:", corrected_string)
    if FINETUNE_AFTER_CORRECTION:
        print("Fine-tuning model using corrected labels...")
        model = finetune_on_corrections(model, rois_28, corrected_list, epochs=FINETUNE_EPOCHS)
        X = np.array(rois_28).astype('float32')/255.0
        X = X.reshape(-1,28,28,1)
        preds = model.predict(X)
        pred_labels = [int(np.argmax(p)) for p in preds]
        print("After fine-tune prediction:", "".join(str(x) for x in pred_labels))
        for (x,y,w,h),lbl in zip(rois_boxes, pred_labels):
            cv2.rectangle(annotated, (x,y), (x+w,y+h), (255,0,0), 2)
            cv2.putText(annotated, str(lbl), (x, y-6), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,0,0), 2)
    out_path = os.path.splitext(img_path)[0] + "_recognized_final.png"
    cv2.imwrite(out_path, annotated)
    print("Annotated result saved to:", out_path)
    print("Final recognized string (after correction):", corrected_string)
    plt.figure(figsize=(8,6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title("Final annotated (green=initial, blue=after-finetune)")
    plt.axis('off')
    plt.show()

Using image: /content/output.jpeg
Initial boxes found: 3
Final boxes after splitting: 3
No model found. Training base CNN on MNIST (epochs = 3 )


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/3
1875/1875 - 61s - 32ms/step - accuracy: 0.9513 - loss: 0.1606 - val_accuracy: 0.9841 - val_loss: 0.0495
Epoch 2/3
1875/1875 - 59s - 32ms/step - accuracy: 0.9822 - loss: 0.0575 - val_accuracy: 0.9899 - val_loss: 0.0302
Epoch 3/3
1875/1875 - 80s - 43ms/step - accuracy: 0.9873 - loss: 0.0398 - val_accuracy: 0.9900 - val_loss: 0.0279


In [3]:
# Create a dummy image file for testing
import cv2
import numpy as np

# Create a blank white image
dummy_image = np.ones((100, 200, 3), dtype=np.uint8) * 255

# Add some text (like a digit) to the image
cv2.putText(dummy_image, '123', (50, 70), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 0), 3)

# Save the image
cv2.imwrite('/content/output.jpeg', dummy_image)
print("Dummy image created at /content/output.jpeg")

Dummy image created at /content/output.jpeg
